# StreamForge — FLUX.1-dev in Pure Rust on Kaggle

Full BF16 FLUX.1-dev inference pipeline written in **pure Rust** using [candle](https://github.com/huggingface/candle).  
Streams the 24GB transformer **block-by-block** from disk through CPU RAM to GPU — peak VRAM stays far below model size because the notebook never loads the whole transformer into RAM at once.

**Source code:** https://github.com/madtunebk/GPULoad

## What this notebook does
1. Copies the converted model files into `/kaggle/working/streamforge/models` to avoid slow dataset-mount reads during streamed inference
2. Downloads CLIP-L + T5-XXL weights from HuggingFace (~10GB, ~2 min)
3. Builds the Rust binaries on Kaggle with CUDA enabled
4. Runs the full pipeline directly from `/kaggle/working/streamforge`: text_encoder → flux_gpu → vae_decode → PNG

## Required Kaggle Dataset
Add your dataset (containing `flux_candle.safetensors` and `vae_candle.safetensors`) via:  
**Add Data → Models → flux-dev-rust-candle** (by valicornea84)

It will be mounted at `/kaggle/input/models/valicornea84/flux-dev-rust-candle/other/default/1/`

## Why RAM looks low on T4

Seeing only ~8 GB RAM used is normal for this project. `flux_gpu` memory-maps the transformer and streams one block at a time, so low RAM usage does **not** mean the full model failed to load. The slow part on Kaggle is usually the storage path: reading streamed weights from the dataset mount is slower than reading local copies from `/kaggle/working`.

## GPU
Enable GPU accelerator: **Settings → Accelerator → GPU T4 x2** (or P100)

In [ ]:
# Verify GPU is available
import os

gpu_info = os.popen("nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader").read()
print(gpu_info)

print(f"HOME={os.environ.get('HOME')}")
print("CUDA available:", os.path.exists('/usr/local/cuda'))

## Step 1 — Check model dataset is mounted

## Step 1b — HuggingFace Authentication

FLUX.1-dev is a **gated model** — you must:
1. Accept the license at https://huggingface.co/black-forest-labs/FLUX.1-dev
2. Create a **classic** Read token at https://huggingface.co/settings/tokens  
   ⚠️ Fine-grained tokens block gated repos by default — use classic OR enable "Access to public gated repositories" in the fine-grained token settings
3. Add it as a Kaggle Secret: **Add-ons → Secrets → Add** with name `HF_TOKEN`

In [ ]:
from kaggle_secrets import UserSecretsClient
import os

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token  # older HF versions
print("HF token loaded")

In [ ]:
import os

DATASET_DIR = "/kaggle/input/models/valicornea84/flux-dev-rust-candle/other/default/1"
FLUX_MODEL  = f"{DATASET_DIR}/flux_candle.safetensors"
VAE_MODEL   = f"{DATASET_DIR}/vae_candle.safetensors"

for path in [FLUX_MODEL, VAE_MODEL]:
    size = os.path.getsize(path) / 1e9
    print(f"OK  {path}  ({size:.1f} GB)")

## Step 2 — Download CLIP-L + T5-XXL from HuggingFace

These weights stay in the HF cache at `~/.cache/huggingface/` which matches the hardcoded paths in the Rust binaries.

> Takes ~2 minutes on Kaggle's network.

In [ ]:
!pip install -q huggingface_hub

from huggingface_hub import snapshot_download
import os

HF_SNAPSHOT_DIR = snapshot_download(
    repo_id="black-forest-labs/FLUX.1-dev",
    allow_patterns=["text_encoder/**", "text_encoder_2/**", "tokenizer/**", "tokenizer_2/**"],
    token=os.environ["HF_TOKEN"],
    ignore_patterns=["*.bin"],
    resume_download=True,
    local_dir_use_symlinks=False,
 )
HF_REPO_DIR = os.path.dirname(os.path.dirname(HF_SNAPSHOT_DIR))
os.environ["STREAMFORGE_HF_REPO_DIR"] = HF_REPO_DIR

print("HF snapshot:", HF_SNAPSHOT_DIR)
print("HF repo dir:", HF_REPO_DIR)
print("Done.")

## Step 3 — Set up workspace, clone repo, and wire model paths

In [ ]:
import os, shutil

WORKDIR = "/kaggle/working/streamforge"
REPO_DIR = "/kaggle/working/GPULOAD"
MODEL_DIR = f"{WORKDIR}/models"
TEMP_DIR = f"{WORKDIR}/temp"
LORA_DIR = f"{WORKDIR}/loras"
BIN_DST = f"{WORKDIR}/bin"

for path in [WORKDIR, MODEL_DIR, f"{MODEL_DIR}/tokenizer", TEMP_DIR, LORA_DIR, BIN_DST]:
    os.makedirs(path, exist_ok=True)

if not os.path.exists(f"{REPO_DIR}/Cargo.toml"):
    !git clone --depth 1 https://github.com/madtunebk/GPULoad.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
print(f"repo    {REPO_DIR}")

for fname, src in [("flux_candle.safetensors", FLUX_MODEL), ("vae_candle.safetensors", VAE_MODEL)]:
    dst = f"{MODEL_DIR}/{fname}"
    shutil.copy2(src, dst)
    size_gb = os.path.getsize(dst) / 1e9
    print(f"copied  {dst}  ({size_gb:.1f} GB)")

tokenizer_candidates = [
    os.path.join(REPO_DIR, "models", "tokenizer", "tokenizer.json"),
    os.path.join(HF_SNAPSHOT_DIR, "tokenizer", "tokenizer.json"),
]
clip_tokenizer_src = next((path for path in tokenizer_candidates if os.path.exists(path)), None)
if clip_tokenizer_src is None:
    raise FileNotFoundError("Could not find tokenizer.json in repo or HF snapshot")

workspace_tokenizer_dst = f"{MODEL_DIR}/tokenizer/tokenizer.json"
os.makedirs(os.path.dirname(workspace_tokenizer_dst), exist_ok=True)
shutil.copy2(clip_tokenizer_src, workspace_tokenizer_dst)
print(f"copied  {workspace_tokenizer_dst}")

repo_tokenizer_dst = os.path.join(REPO_DIR, "models", "tokenizer", "tokenizer.json")
os.makedirs(os.path.dirname(repo_tokenizer_dst), exist_ok=True)
shutil.copy2(clip_tokenizer_src, repo_tokenizer_dst)
print(f"copied  {repo_tokenizer_dst}")

os.environ["STREAMFORGE_MODEL_DIR"] = MODEL_DIR
os.environ["STREAMFORGE_HF_REPO_DIR"] = HF_REPO_DIR
print(f"STREAMFORGE_MODEL_DIR={MODEL_DIR}")
print(f"STREAMFORGE_HF_REPO_DIR={HF_REPO_DIR}")
print("workspace tokenizer exists:", os.path.exists(workspace_tokenizer_dst))
print("repo tokenizer exists:", os.path.exists(repo_tokenizer_dst))

## Step 4 — Build from source on Kaggle

This uses the simple workflow that already worked for you: update `/kaggle/working/GPULOAD`, build there, then copy the fresh binaries into `/kaggle/working/streamforge/bin`.

The dataset symlinks were removed on purpose. For `flux_gpu`, local copies in `/kaggle/working/streamforge/models` are usually faster than reading through the Kaggle dataset mount while the transformer is streamed block-by-block.

In [ ]:
import os, shutil

repo_dir = REPO_DIR
BIN_DST = f"{WORKDIR}/bin"

os.chdir("/kaggle/working/GPULOAD")
!git pull

os.chdir("/kaggle/working/GPULOAD")
!cargo build --release --features cuda

for b in ["text_encoder", "flux_gpu", "vae_decode"]:
    shutil.copy2(f"{repo_dir}/target/release/{b}", f"{BIN_DST}/{b}")
    os.chmod(f"{BIN_DST}/{b}", 0o755)
    print(f"copied  {BIN_DST}/{b}")

!{BIN_DST}/flux_gpu --help | head -1

## Step 5 — Run the pipeline

Edit `PROMPT`, `WIDTH`, `HEIGHT`, `STEPS`, and `SEED` below.

Each command below runs directly after `os.chdir("/kaggle/working/streamforge")`, with no `subprocess` wrapper and no Python-built shell command strings.

In [ ]:
import os

checks = [
    MODEL_DIR,
    f"{MODEL_DIR}/flux_candle.safetensors",
    f"{MODEL_DIR}/vae_candle.safetensors",
    f"{MODEL_DIR}/tokenizer/tokenizer.json",
    TEMP_DIR,
    BIN_DST,
    f"{BIN_DST}/text_encoder",
    f"{BIN_DST}/flux_gpu",
    f"{BIN_DST}/vae_decode",
    HF_REPO_DIR,
    os.path.join(HF_REPO_DIR, "snapshots"),
]

all_ok = True
for p in checks:
    exists = os.path.exists(p)
    status = "OK " if exists else "MISSING"
    if not exists:
        all_ok = False
    print(f"[{status}] {p}")

if not all_ok:
    raise RuntimeError("Pre-flight check failed — fix missing paths above before running pipeline")

print(f"\nSTREAMFORGE_MODEL_DIR={MODEL_DIR}")
print(f"STREAMFORGE_HF_REPO_DIR={HF_REPO_DIR}")
print("All checks passed.")

In [ ]:
PROMPT = "an ivory unicorn with a molten-gold horn and a pearl-white mane, standing knee-deep in moonlit water, liquid starlight dripping from its mane, a halo of fireflies and blue will-o-wisps swirling around it, gilded filigree armor on its chest, cathedral-tall ancient forest in the background, cinematic volumetric light, luminous mist, high fantasy oil painting, lavish detail, regal aura, masterpiece"
WIDTH = 768
HEIGHT = 1024
STEPS = 20
GUIDANCE = 3.5
SEED = 42
DTYPE = "f16"
LORA = None
LORA_SCALE = 1.0

In [ ]:
import os

PROMPT_EMBEDS = f"{TEMP_DIR}/prompt_embeds.safetensors"
LATENTS_PATH = f"{TEMP_DIR}/latents.safetensors"
OUTPUT_PATH = f"{TEMP_DIR}/output_rust.png"

os.environ["PATH"] = f"{BIN_DST}:{os.environ['PATH']}"
os.environ["STREAMFORGE_MODEL_DIR"] = MODEL_DIR
os.environ["STREAMFORGE_HF_REPO_DIR"] = HF_REPO_DIR
os.environ["STREAMFORGE_DTYPE"] = DTYPE
os.chdir("/kaggle/working/streamforge")

for stale_path in [PROMPT_EMBEDS, LATENTS_PATH, OUTPUT_PATH]:
    if os.path.exists(stale_path):
        os.remove(stale_path)
        print(f"removed stale file: {stale_path}")

print("Environment ready")
print(f"cwd={os.getcwd()}")
print(f"STREAMFORGE_MODEL_DIR={MODEL_DIR}")
print(f"STREAMFORGE_HF_REPO_DIR={HF_REPO_DIR}")
print(f"STREAMFORGE_DTYPE={DTYPE}")
print(f"PROMPT_EMBEDS={PROMPT_EMBEDS}")
print(f"LATENTS_PATH={LATENTS_PATH}")
print(f"OUTPUT_PATH={OUTPUT_PATH}")
print("tokenizer exists:", os.path.exists("models/tokenizer/tokenizer.json"))
print("flux model size (GB):", round(os.path.getsize(f"{MODEL_DIR}/flux_candle.safetensors") / 1e9, 2))
print("Low RAM usage is expected: the transformer is memory-mapped and streamed block-by-block.")

In [ ]:
os.chdir("/kaggle/working/streamforge")
print(os.getcwd())

if LORA:
    print("LORA is set. Add --lora and --lora-scale manually to the command cells below.")

In [ ]:
os.chdir("/kaggle/working/streamforge")
!./bin/text_encoder \
  "$PROMPT" \
  --device cuda

print("prompt embeds exists:", os.path.exists(PROMPT_EMBEDS))

In [ ]:
os.chdir("/kaggle/working/streamforge")
!/kaggle/working/streamforge/bin/flux_gpu \
  --embeddings {PROMPT_EMBEDS} \
  --out-latents {LATENTS_PATH} \
  --width {WIDTH} \
  --height {HEIGHT} \
  --steps {STEPS} \
  --guidance {GUIDANCE} \
  --seed {SEED}

print("latents exist:", os.path.exists(LATENTS_PATH))

In [ ]:
os.chdir("/kaggle/working/streamforge")
!/kaggle/working/streamforge/bin/vae_decode \
  --latents {LATENTS_PATH} \
  --out {OUTPUT_PATH}

print("output exists:", os.path.exists(OUTPUT_PATH))

## Result

In [ ]:
from IPython.display import Image, display
import shutil, os

output_src = f"{WORKDIR}/temp/output_rust.png"
output_dst = "/kaggle/working/output_rust.png"
shutil.copy2(output_src, output_dst)

display(Image(filename=output_dst))
print(f"Saved to {output_dst}")